# 🔍 Hallucination Detection in Small Language Models

## Evaluation Notebook

This notebook runs the full evaluation pipeline for the SMILES 2026 challenge.

### Pipeline overview

1. **Load dataset** — read the CSV and build input texts
2. **Extract features** — run a single LLM forward pass over all samples and apply your aggregation (run once, results reused)
3. **Define splits** ← **edit this cell** — choose a splitting strategy (random split, k-fold, group-aware, …)
4. **Train & evaluate** — train `HallucinationProbe` on each split and report metrics
5. **Save results** — write a JSON summary

### Files you can edit

| File | What to implement |
|------|-------------------|
| `aggregation.py` | Layer selection and token-pooling strategy |
| `probe.py` | Probe classifier (`nn.Module` subclass) |
| **This notebook** | Splitting strategy and evaluation loop |

> **Do not edit** `model.py` or `dataset.py` — these are fixed infrastructure.

## 1 · Setup

In [ ]:
# Uncomment the line below when running in Google Colab for the first time.
# !pip install -r requirements.txt

In [ ]:
import json
import time

import numpy as np
import torch
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from aggregation import aggregate
from model import get_model_and_tokenizer, extract_hidden_states
from probe import HallucinationProbe

## 2 · Configuration

Adjust `DATA_FILE`, `OUTPUT_FILE`, and `BATCH_SIZE` as needed.

In [ ]:
DATA_FILE   = "./data/dataset.csv"   # path to the dataset CSV (auto-created if absent)
OUTPUT_FILE = "results.json"         # where to write the results summary
BATCH_SIZE  = 4                      # reduce to 1–2 if you run out of GPU memory

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device : {device}")
print(f"Data   : {DATA_FILE}")

## 3 · Load dataset

In [ ]:
df = pd.read_csv(DATA_FILE)
all_texts  = [f"{row['prompt']}{row['response']}" for _, row in df.iterrows()]
all_labels = np.array([int(float(h)) for h in df["label"]])

n_total = len(all_labels)
print(f"Loaded {n_total} samples  "
      f"({all_labels.sum()} hallucinated / {(all_labels == 0).sum()} truthful)")

## 4 · Feature extraction

Hidden states are extracted **once** for the entire dataset and aggregated on-the-fly
to save memory.  The resulting feature matrix `X` is reused across all splits.

> ⚠️ This step loads the LLM and runs a forward pass — it may take a few minutes.

In [ ]:
model, tokenizer = get_model_and_tokenizer()

t0 = time.time()
all_features, _ = extract_hidden_states(
    model,
    tokenizer,
    all_texts,
    device=device,
    batch_size=BATCH_SIZE,
    aggregate_fn=aggregate,   # applied in-place to save memory
)
extract_time = time.time() - t0
print(f"Extraction done in {extract_time:.1f} s")

# Stack feature vectors into a (n_samples, feature_dim) NumPy matrix.
X = np.vstack([h.numpy() for h in all_features])   # shape: (N, feature_dim)
y = all_labels                                       # shape: (N,)

print(f"Feature matrix : {X.shape}  (feature_dim = {X.shape[1]})")

---

## 5 · ⭐ Splitting strategy  ← **edit this cell**

Define how the dataset is divided into train / validation / test subsets.

Your splitting code must produce a Python list called **`splits`** where each
element is a `(idx_train, idx_val, idx_test)` tuple of integer index arrays
into `X` and `y`:

```
splits: list[tuple[np.ndarray, np.ndarray | None, np.ndarray]]
```

Set `idx_val = None` if you do not need a validation fold (e.g. pure k-fold).

The evaluation cell below iterates over every `(train, val, test)` triple and
averages the metrics, so both a single split and k-fold work without changes.

### Ideas to explore
- **Stratified random split** (default below)
- **Stratified k-fold** — rotate the test set; average metrics across k folds
- **Group-aware split** — keep all rows with the same `question` in the same fold
  so the probe cannot memorise question wording
- **Time-ordered split** — hold out the highest `id` values to simulate deployment

In [ ]:
# -----------------------------------------------------------------------
# STUDENT: Replace or extend the splitting strategy below.
# The only requirement: produce a list called `splits` where each element
# is a (idx_train, idx_val, idx_test) tuple of integer index arrays.
# -----------------------------------------------------------------------

from sklearn.model_selection import train_test_split

# --- Option A: Simple stratified random split (default) -----------------
TEST_SIZE    = 0.15
VAL_SIZE     = 0.15
RANDOM_STATE = 42

idx = np.arange(len(X))

idx_train_val, idx_test = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
relative_val = VAL_SIZE / (1.0 - TEST_SIZE)
idx_train, idx_val = train_test_split(
    idx_train_val,
    test_size=relative_val,
    random_state=RANDOM_STATE,
    stratify=y[idx_train_val],
)

splits = [(idx_train, idx_val, idx_test)]

print(f"Split  : train={len(idx_train)} | val={len(idx_val)} | test={len(idx_test)}")

# --- Option B: Stratified k-fold (uncomment to use instead) -------------
# from sklearn.model_selection import StratifiedKFold
#
# K = 5
# kfold = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)
# splits = []
# for fold_train, fold_test in kfold.split(X, y):
#     splits.append((fold_train, None, fold_test))   # no separate val fold
#
# print(f"K-Fold : {K} folds  ({len(splits[0][0])} train / {len(splits[0][2])} test per fold)")

# --- Option C: Group-aware split (keep same question in same fold) ------
# from sklearn.model_selection import GroupShuffleSplit
#
# groups = df["question"].astype(str).values   # group identifier per sample
# gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=42)
# idx_train_val, idx_test = next(gss.split(X, y, groups=groups))
# idx_train, idx_val = train_test_split(
#     idx_train_val, test_size=VAL_SIZE / (1.0 - TEST_SIZE),
#     random_state=42, stratify=y[idx_train_val],
# )
# splits = [(idx_train, idx_val, idx_test)]

---

## 6 · Evaluation

The loop below iterates over every `(train, val, test)` triple in `splits`,
trains the majority-class baseline and the `HallucinationProbe`, and records
Accuracy, F1, and AUROC for each fold.

In [ ]:
def _fmt(value: float) -> str:
    """Format a [0, 1] metric value as a percentage string."""
    return f"{value * 100:.2f}%"


def evaluate_probe(probe, X, y, idx_train, idx_val, idx_test):
    """Train *probe* and return a metrics dict for val + test splits.

    Args:
        probe:      A freshly instantiated ``HallucinationProbe``.
        X:          Full feature matrix of shape ``(N, feature_dim)``.
        y:          Full label array of shape ``(N,)``.
        idx_train:  Integer indices for the training subset.
        idx_val:    Integer indices for the validation subset, or ``None``.
        idx_test:   Integer indices for the test subset.

    Returns:
        A dict containing accuracy, F1, and AUROC for the available splits.
    """
    probe.fit(X[idx_train], y[idx_train])

    results = {}

    for split_name, idx_split in [("val", idx_val), ("test", idx_test)]:
        if idx_split is None:
            continue
        y_true  = y[idx_split]
        y_pred  = probe.predict(X[idx_split])
        y_prob  = probe.predict_proba(X[idx_split])[:, 1]

        results[f"{split_name}_accuracy"] = accuracy_score(y_true, y_pred)
        results[f"{split_name}_f1"]       = f1_score(y_true, y_pred, zero_division=0)
        try:
            results[f"{split_name}_auroc"] = roc_auc_score(y_true, y_prob)
        except ValueError:
            results[f"{split_name}_auroc"] = float("nan")

    return results

In [ ]:
fold_results = []

for fold_idx, (idx_train, idx_val, idx_test) in enumerate(splits):
    fold_label = f"Fold {fold_idx + 1}/{len(splits)}"
    print(f"\n{'─' * 50}")
    print(f"  {fold_label}  —  "
          f"train={len(idx_train)}  "
          f"val={len(idx_val) if idx_val is not None else 'N/A'}  "
          f"test={len(idx_test)}")
    print(f"{'─' * 50}")

    # ── Checkpoint 1: Majority-class baseline ──────────────────────────
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X[idx_train], y[idx_train])
    y_dummy      = dummy.predict(X[idx_test])
    baseline_acc = accuracy_score(y[idx_test], y_dummy)
    baseline_f1  = f1_score(y[idx_test], y_dummy, zero_division=0)
    print(f"  Baseline  — Acc: {_fmt(baseline_acc)}  F1: {_fmt(baseline_f1)}")

    # ── Checkpoint 2/3: Student probe ─────────────────────────────────
    probe   = HallucinationProbe()
    metrics = evaluate_probe(probe, X, y, idx_train, idx_val, idx_test)

    if "val_auroc" in metrics:
        print(f"  Probe val  — Acc: {_fmt(metrics['val_accuracy'])}  "
              f"F1: {_fmt(metrics['val_f1'])}  "
              f"AUROC: {_fmt(metrics['val_auroc'])}")
    print(f"  Probe test — Acc: {_fmt(metrics['test_accuracy'])}  "
          f"F1: {_fmt(metrics['test_f1'])}  "
          f"AUROC: {_fmt(metrics['test_auroc'])}")

    fold_results.append({
        "fold": fold_idx + 1,
        "n_train": len(idx_train),
        "n_val":   len(idx_val) if idx_val is not None else 0,
        "n_test":  len(idx_test),
        "baseline_accuracy": baseline_acc,
        "baseline_f1":       baseline_f1,
        **metrics,
    })

## 7 · Summary and results

In [ ]:
def _nanmean(values):
    valid = [v for v in values if v == v]   # filter NaN
    return float(np.mean(valid)) if valid else float("nan")


avg_baseline_acc  = _nanmean([r["baseline_accuracy"]  for r in fold_results])
avg_baseline_f1   = _nanmean([r["baseline_f1"]        for r in fold_results])
avg_test_acc      = _nanmean([r["test_accuracy"]       for r in fold_results])
avg_test_f1       = _nanmean([r["test_f1"]             for r in fold_results])
avg_test_auroc    = _nanmean([r["test_auroc"]           for r in fold_results])
avg_val_auroc     = _nanmean([r.get("val_auroc", float("nan")) for r in fold_results])

W = 60
print("\n" + "=" * W)
print(" Hallucination Detection — Evaluation Summary")
if len(fold_results) > 1:
    print(f" (averaged over {len(fold_results)} folds)")
print("=" * W)
print(f"  {'Checkpoint':<35} {'Accuracy':>9} {'F1':>7} {'AUROC':>7}")
print("-" * W)
print(f"  {'1. Majority-class baseline':<35} {_fmt(avg_baseline_acc):>9} {_fmt(avg_baseline_f1):>7} {'N/A':>7}")
if avg_val_auroc == avg_val_auroc:   # not NaN → val split exists
    avg_val_acc = _nanmean([r.get("val_accuracy", float("nan")) for r in fold_results])
    avg_val_f1  = _nanmean([r.get("val_f1",       float("nan")) for r in fold_results])
    print(f"  {'2. Probe (val split)':<35} {_fmt(avg_val_acc):>9} {_fmt(avg_val_f1):>7} {_fmt(avg_val_auroc):>7}")
print(f"  {'3. Probe (test split)':<35} {_fmt(avg_test_acc):>9} {_fmt(avg_test_f1):>7} {_fmt(avg_test_auroc):>7}")
print("-" * W)
print(f"  Feature dim  : {X.shape[1]}")
print(f"  Total samples: {len(X)}")
print(f"  Folds        : {len(fold_results)}")
print(f"  Extract time : {extract_time:.1f} s")
print("=" * W)
print()
print(f"★  Primary metric — Test AUROC: {_fmt(avg_test_auroc)}")

# ── Save results ────────────────────────────────────────────────────────
summary = {
    "folds":            fold_results,
    "avg_baseline_accuracy": avg_baseline_acc,
    "avg_baseline_f1":       avg_baseline_f1,
    "avg_val_auroc":         avg_val_auroc,
    "avg_test_accuracy":     avg_test_acc,
    "avg_test_f1":           avg_test_f1,
    "avg_test_auroc":        avg_test_auroc,
    "feature_dim":           int(X.shape[1]),
    "n_samples":             int(len(X)),
    "n_folds":               len(fold_results),
    "extract_time_s":        extract_time,
}

with open(OUTPUT_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nResults saved to '{OUTPUT_FILE}'")